# Data Engineering Layer


In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\Shramanth P Acharya\Unified Mentor Private\coffee_demand_forecasting\data\coffee_sales.csv.csv")

df.columns = df.columns.str.lower()
df['transaction_time'] = pd.to_datetime(df['transaction_time'])

df['revenue'] = df['transaction_qty'] * df['unit_price']

C:\Users\Shramanth P Acharya\AppData\Local\Temp\ipykernel_29232\2939883070.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['transaction_time'] = pd.to_datetime(df['transaction_time'])


# Time-Series Construction

In [2]:
df_hourly = df.groupby(
    ['store_id', pd.Grouper(key='transaction_time', freq='H')]
).agg({
    'transaction_qty':'sum',
    'revenue':'sum'
}).reset_index()

def complete_time_index(store_df):
    idx = pd.date_range(
        start=store_df['transaction_time'].min(),
        end=store_df['transaction_time'].max(),
        freq='H'
    )
    store_df = store_df.set_index('transaction_time').reindex(idx)
    store_df['store_id'] = store_df['store_id'].iloc[0]
    return store_df.fillna(0).rename_axis('transaction_time').reset_index()

df_hourly = (
    df_hourly.groupby('store_id')
    .apply(complete_time_index)
    .reset_index(drop=True)
)

# Feature Engineering 

In [3]:
df_hourly['hour'] = df_hourly['transaction_time'].dt.hour
df_hourly['day_of_week'] = df_hourly['transaction_time'].dt.dayofweek
df_hourly['month'] = df_hourly['transaction_time'].dt.month
df_hourly['is_weekend'] = (df_hourly['day_of_week'] >= 5).astype(int)

df_hourly['lag_1'] = df_hourly.groupby('store_id')['transaction_qty'].shift(1)
df_hourly['lag_24'] = df_hourly.groupby('store_id')['transaction_qty'].shift(24)
df_hourly['lag_168'] = df_hourly.groupby('store_id')['transaction_qty'].shift(168)

df_hourly['roll_72'] = df_hourly.groupby('store_id')['transaction_qty'] \
                                .rolling(72).mean().reset_index(0,drop=True)

df_hourly['roll_168'] = df_hourly.groupby('store_id')['transaction_qty'] \
                                 .rolling(168).mean().reset_index(0,drop=True)

df_hourly = df_hourly.dropna().reset_index(drop=True)

# Train-Test strategy

In [5]:
split_date = df_hourly['transaction_time'].quantile(0.8)

train = df_hourly[df_hourly['transaction_time'] <= split_date]
test  = df_hourly[df_hourly['transaction_time'] > split_date]

# Model Layer Naive

In [6]:
y_test = test['transaction_qty']
y_pred_naive = test['lag_1']
y_pred_ma = test['roll_72']

# Gradient Boosting

In [8]:
from sklearn.ensemble import GradientBoostingRegressor

features = [
    'hour','day_of_week','month','is_weekend',
    'lag_1','lag_24','lag_168',
    'roll_72','roll_168'
]

model = GradientBoostingRegressor()
model.fit(train[features], train['transaction_qty'])

y_pred_gb = model.predict(test[features])

ValueError: Found array with 0 sample(s) (shape=(0, 9)) while a minimum of 1 is required by GradientBoostingRegressor.

# Evaluation (PRD Metrics)

In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred)
    }

# Peak Demand Detection (Critical KPI)

In [ ]:
threshold = df_hourly['transaction_qty'].quantile(0.9)

test['actual_peak'] = (test['transaction_qty'] >= threshold).astype(int)
test['pred_peak'] = (y_pred_gb >= threshold).astype(int)

peak_capture_rate = (
    (test['actual_peak'] & test['pred_peak']).sum() /
    test['actual_peak'].sum()
)